# Phase 4 — Modeling & Hyperparameter Tuning


In [ ]:
# Phase 4 — Modeling & Hyperparameter Tuning

# ============================================================
# Phase 4 — Setup: import models and metrics
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score,
                             roc_curve, classification_report)

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Libraries ready")

# ============================================================
# Rebuild data and pipeline (shortcut of previous phases)
# ============================================================
DATA_FILE = Path("/kaggle/input/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020/ai4i2020.csv")
df = pd.read_csv(DATA_FILE)

SENSOR_COLUMNS = [
    "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]",
]
BINARY_TARGET = "Machine failure"
FAILURE_TYPE_COLUMNS = ["TWF", "HDF", "PWF", "OSF", "RNF"]
CATEGORICAL_COL = "Type"

flag_sum = df[FAILURE_TYPE_COLUMNS].sum(axis=1)
df["Failure Type"] = "No Failure"
failed_mask = df[BINARY_TARGET] == 1
df.loc[failed_mask, "Failure Type"] = (
    df.loc[failed_mask, FAILURE_TYPE_COLUMNS]
    .apply(lambda row: "+".join(row.index[row == 1]) or "Unknown", axis=1)
)

df["Power [W]"] = df["Torque [Nm]"] * df["Rotational speed [rpm]"] * (2 * np.pi / 60)
df["Temp Diff [K]"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["Overstrain [min·Nm]"] = df["Tool wear [min]"] * df["Torque [Nm]"]
ENGINEERED_COLUMNS = ["Power [W]", "Temp Diff [K]", "Overstrain [min·Nm]"]

feature_cols = SENSOR_COLUMNS + ENGINEERED_COLUMNS + [CATEGORICAL_COL]
X = df[feature_cols]
y = df[BINARY_TARGET]

numeric_cols = [c for c in X.columns if X[c].dtype in ["int64", "float64"]]
categorical_cols = [c for c in X.columns if X[c].dtype == "object"]

numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=[["L", "M", "H"]])),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)

SCALE_POS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()

print(f"✅ Ready: train={X_train.shape}, test={X_test.shape}")
print(f"   scale_pos_weight = {SCALE_POS_WEIGHT:.2f}")

# ============================================================
# Baseline model: Random Forest
# ============================================================
def build_model(preprocessor, classifier):
    """Combine the preprocessor and classifier into a single Pipeline."""
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])

rf_model = build_model(
    preprocessor,
    RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(rf_model, X_train, y_train,
                         cv=cv, scoring="recall", n_jobs=-1)

print("Recall per fold:", [round(s, 3) for s in scores])
print(f"Mean Recall for Random Forest: {scores.mean():.4f} ± {scores.std():.4f}")

# ============================================================
# Advanced model: XGBoost
# ============================================================
xgb_model = build_model(
    preprocessor,
    XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        scale_pos_weight=SCALE_POS_WEIGHT,
        random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1,
    ),
)

scores_xgb = cross_val_score(xgb_model, X_train, y_train,
                             cv=cv, scoring="recall", n_jobs=-1)

print("Recall per fold:", [round(s, 3) for s in scores_xgb])
print(f"Mean Recall for XGBoost: {scores_xgb.mean():.4f} ± {scores_xgb.std():.4f}")

# ============================================================
# Advanced model: LightGBM
# ============================================================
lgbm_model = build_model(
    preprocessor,
    LGBMClassifier(
        n_estimators=300, learning_rate=0.1, num_leaves=31,
        is_unbalance=True, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    ),
)

scores_lgbm = cross_val_score(lgbm_model, X_train, y_train,
                              cv=cv, scoring="recall", n_jobs=-1)

print("Recall per fold:", [round(s, 3) for s in scores_lgbm])
print(f"Mean Recall for LightGBM: {scores_lgbm.mean():.4f} ± {scores_lgbm.std():.4f}")

# ============================================================
# Compare the three models
# ============================================================
results = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost", "LightGBM"],
    "Mean Recall": [scores.mean(), scores_xgb.mean(), scores_lgbm.mean()],
    "Std Dev": [scores.std(), scores_xgb.std(), scores_lgbm.std()],
}).sort_values("Mean Recall", ascending=False)

print(results.round(4).to_string(index=False))

# ============================================================
# Hyperparameter tuning
# ============================================================
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__max_depth": [4, 6, 8],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__n_estimators": [200, 300],
}

grid_search = GridSearchCV(
    xgb_model, param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring="recall", n_jobs=-1, verbose=1,
)

grid_search.fit(X_train, y_train)

print("\n🏆 Best params:", grid_search.best_params_)
print(f"🏆 Best validation Recall: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
